# Load the Nsight data

Note: pipit does not ingest the raw .nsys-rep file format. You must pass pipit the
sqlite file generated from the nsys-rep file

In [2]:
import pipit

trace = pipit.Trace.from_nsight_sqlite("llama3-8b-instruct-decode.sqlite")

/Users/thomasli/pipit/pipit/readers/nsight_sqlite_reader.py:219: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  trace_df = pd.concat(traces, axis=0)


In [4]:
# Generate a flat profile
# This is like the Nsight CUDA API/Kernel Summary except its all combined into on result series
trace.flat_profile().sort_values(by="time.exc", ascending=False)

,time.exc
Name,
ampere_bf16_s16816gemm_bf16_64x64_sliced1x2_ldg8_f2f_stages_64x5_tn,15602385.0
ampere_bf16_s16816gemm_bf16_64x64_sliced1x2_ldg8_f2f_stages_64x6_tn,8141949.0
flash_fwd_splitkv_kernel,793758.0
rotary_kernel,394593.0
flash_fwd_splitkv_combine_kernel,366904.0
triton_red_fused__softmax__to_copy_argmax_div_exponential_masked_fill_23,327136.0
triton_poi_fused_mul_silu_7,190752.0
DeviceRadixSortDownsweepKernel,162848.0
triton_red_fused__to_copy_add_mean_mul_rsqrt_12,157312.0


# Time breakdowns

Note: This is non-overlap aware right now

In [ ]:
# allows to zoom into an annotation on trace
# returns a trace (you can call trace methods like flat_profile again on the returned trace to see
# the most time consuming functions within an annotation)
trace.filter_by_label("Torch-Compiled Region, op_id = 81")

# Calculates time spent both on CPU and GPU
# in every annotation found inside the trace
trace.time_breakdown()

# Calculates idle time within the trace
# use idle_functions kwarg to specify names
# of events that should be clasified as idle time
# (e.g. cudaDeviceSynchronize).
# note: this doesn't support using a regex for idle_functions as of right now
trace.idle_time()

# To access the dataframe inside a trace object directly
trace.events
# FYI: you can then filter name with a regex using
pat = "nccl"
events = trace.events[trace.events["Name"].str.startswith(pat)]
# to repack it up into a trace
trace = pipit.trace.Trace(None, events, trace.parallelism_levels)